In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
SERPER_API_KEY=os.getenv("SERPER_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")

In [3]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = LANGCHAIN_PROJECT

### ReAct Agent - Synergizing Reasonsing + Action / Acting

src - https://arxiv.org/abs/2210.03629

React agents and function calling agents are two distinct approaches to leveraging Large Language Model(LLMs) for automating tasks. Both methods aim to integrate LLMs with external tools and systems, but they differ in their underlying logic and implementation.

#### React Agent

A React agent combines reasoning and action in a continuous loop. the LLM "thinks" about the problem, decides the steps to be taken, allows the agent to take action based on its reasoning, and then observes the results to refine its understanding. This approach relies on "ReAct prompting," which involves crafting prompts that guide the LLM to alternate between reasoning and action steps.

**Key features of React agents:**

**1.Reasoning and action loop:** The agent continuously reasons about the task, decides on actions, and observes the outcomes to refine its understanding.

**2.ReACT prompting:** The agent uses specific prompting techniques to guide the LLM’s reasoning and action steps.

**3.Agent types:** React agents are tailored for specific tasks and utilize CoT (Conversational Task) and ReACT prompting.

**4.Strengths:** Suitable for tasks requiring multi-step reasoning, complex planning, and understanding of context.

**5.Limitations:** Can be computationally expensive due to the reasoning steps involved and requires more effort to define prompts and actions.

#### Function Calling Agent

A function calling agent, introduced by OpenAI, allows LLMs to understand a pre-defined format of a function (JSON structure) as part of a user prompt and intelligently choose to output a JSON object containing arguments to call a function (tool in an agent). This approach relies on pre-defined functions and their parameters.

Key features of function calling agents:

**1.Pre-defined functions:** The agent uses pre-defined functions with well-defined parameters and outputs.

**2.JSON-based input:** The agent receives JSON objects containing function names and arguments.

**3.Agent simplicity:** Function calling agents are generally easier to implement, relying on pre-defined functions and parameters.

**4.Strengths:** Excels at integrating LLMs with external systems and performing specific tasks through well-defined functions.

**5.Limitations:** Less suitable for open-ended tasks where actions are not pre-defined. Comparison

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langchain.agents import create_agent # modern agent   

C:\Users\rootl\AppData\Local\Temp\ipykernel_18664\2243701508.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


In [5]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [6]:
# Create a Google Search tool using the Google Serper API Wrapper
google_search = GoogleSerperAPIWrapper(serper_api_key=os.getenv("SERPER_API_KEY"))

In [7]:
# Convert search into a LangChain tool

@tool
def google_search_tool(query: str) -> str:
    """
    Search the web for current, real-time, or
    factual information.

    Use this tool when the answer requires
    information from the internet.
    """
    return google_search.run(query)

In [8]:
agent = create_agent(
        model=llm,
        tools=[google_search_tool],

        system_prompt="""
        You are a helpful and accurate AI assistant.

        You have access to a Google web search tool.

        Use the google_search_tool when:
        - The user asks for current information.
        - The user asks for real-time information.
        - The information may have changed recently.
        - You need to verify factual information.
        - You do not know the answer confidently.

        After getting the search results, use them to
        formulate the final answer.

        Do not mention internal tool calls.
        Give a concise and accurate answer.

    """
)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "What is the current price of NVIDIA (NVDA) stock, and how much has it changed today?"
            )
        }
    ]
})

print(response["messages"][-1].content)

In [15]:
import requests
import os

url = "https://google.serper.dev/search"

headers = {
    "X-API-KEY": os.getenv("SERPER_API_KEY"),
    "Content-Type": "application/json"
}

payload = {
    "q": "NVIDIA stock price today"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 403
Response: {"message":"Unauthorized.","statusCode":403}


### ReAct Agent with Custom tools

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

In [20]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [21]:
# EMPLOYEE ID TOOL

@tool
def get_employee_id(name: str) -> str:
    """
    Get the employee ID for a given employee name.

    Args:
        name: Name of the employee.
    """

    fake_employees = {
        "Alice": "E001",
        "Bob": "E002",
        "Charlie": "E003",
        "Diana": "E004",
        "Evan": "E005",
        "Fiona": "E006",
        "George": "E007",
        "Hannah": "E008",
        "Ian": "E009",
        "Jasmine": "E010"
    }

    return fake_employees.get(
        name,
        "Employee not found"
    )

In [22]:
@tool
def get_employee_salary(employee_id: str) -> str:
    """
    Get the salary of an employee using their employee ID.

    Args:
        employee_id: Employee ID.
    """

    employee_salaries = {
        "E001": 56000,
        "E002": 47000,
        "E003": 52000,
        "E004": 61000,
        "E005": 45000,
        "E006": 58000,
        "E007": 49000,
        "E008": 53000,
        "E009": 50000,
        "E010": 55000
    }

    salary = employee_salaries.get(employee_id)

    if salary is None:
        return "Employee not found"

    return f"${salary:,}"

In [23]:
tools = [
    get_employee_id,
    get_employee_salary
]

agent = create_agent(
    model=llm,
    tools=tools,

    system_prompt="""
    You are an employee information assistant.

    You have two tools:

    1. get_employee_id
       Use this tool to find an employee ID from an employee name.

    2. get_employee_salary
       Use this tool to find an employee salary from an employee ID.

    When the user asks for an employee's salary:

    Step 1:
    Find the employee ID using get_employee_id.

    Step 2:
    Use that employee ID with get_employee_salary.

    Step 3:
    Return the employee's salary clearly.

    If the employee does not exist, say that the employee
    was not found.

    Do not expose internal tool calls.
    """
)

In [24]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the salary of Evan?"
        }
    ]
})


c:\Users\rootl\OneDrive\Documents\euron\YT\sunny_savita\end-to-end-langGraph\venv\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\rootl\OneDrive\Documents\euron\YT\sunny_savita\end-to-end-langGraph\venv\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\rootl\OneDrive\Documents\euron\YT\sunny_savita\end-to-end-langGraph\venv\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [26]:
print(response["messages"][-1].text)

The salary of Evan is $45,000.


In [27]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the salary of Joe Wilson?"
        }
    ]
})

c:\Users\rootl\OneDrive\Documents\euron\YT\sunny_savita\end-to-end-langGraph\venv\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\rootl\OneDrive\Documents\euron\YT\sunny_savita\end-to-end-langGraph\venv\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [28]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Joe Wilson was not found in the employee records.', 'extras': {'signature': 'EoQDCoEDARFNMg/XUAPZ0ohAQrEn6kEjPdL+yMic62dutazXJUXOVYjK+633U6zCLi63MfHRBIfWmTo5JiPAsJOY9lb1eSYn09TwGCHMz1XUkYp21YO64Yy2sOJQ3TW1lSWpql498EZbjVstThJv/bTBmzkZYUW1sWPDjlR1YRTWdCAxyjioPqTKTU+TD1bSUlcbw4dMQPPgHp9RS1A+azvSb5Br8kwgq6ENzFMvItkTmUVIg/OFdSicSHdBD03Fi8N5TkDaEu5VEdsA/g6RB5pKQgQl7sW9FfJAecty2qbFLx/ieWdC1NEGqtJ3Nk7NV6nqhtio4Gn3Dm8GwkDIbaC1h0pGkm3q77pNa6hw8YpNcStkm8nZOrDscYn1PXc6YBhDeRBdkCEW5ZHHhMSCI2PRsmZLPgwngMcgDx9MHm+E0G459uO/Gi5aU05JJCuirJucFa9LoM/fys0si0nNsI9TVXlWbHS2g42cDEguzI0wQvxhIIeetBQu5nsbhlhf6V2tF5J7AQ=='}}]
